# Explaining Nearest-Neighbor Classifiers: Weighted and Thresholded Classifiers

In the [previous notebook](explainers_1.ipynb) we only considered unweighted $k$-nearest neighbor classifiers.
Next, let's focus on the slightly more advanced _weighted_ $k$-nearest neighbor classifier.

Note: Please make sure to read the previous notebook first, since we explain the basics concepts and introduce some mathematical notation there.

## Weighted $k$-Nearest Neighbor Classification

Weighted $k$-nearest neighbor (WKNN) classifiers work similarly to their unweighted counterparts, but assign a weight $w_i$ to each training data point $x_i$, which is inversely proportional to the distance $d(x_i, x)$ from the point $x$ being predicted. A common choice for the weight function is $w_i := \frac{1}{d(x_i, x)}$. This is also the only weight function we support.

When calculating the probability of predicting some class $c \in C$, the model then considers the weighted proportion of training points with class $c$ among the $k$-nearest neighbors of $x$:
$$
    \hat y = \underset{c\in C}{\text{argmax}} \frac{
        \sum_{j=1}^{k} w_{\alpha_j} \, \chi(y_{\alpha_j}=c)
    }{
        \sum_{j=1}^{k} w_{\alpha_j}
    },
$$

where $\alpha_j$ is the index of the $j$-nearest training data point to $x$. Next, we'll discuss how to design a utility function for efficiently calculating Shapley Valuse for WKNN models. If you're only interested in the actual code, you can of course skip this section.

## Making WKNN Shapley-friendly

In order to make WKNN models more suitable for efficiently computing Shapley Values, some adjustments are made compared to unweighted KNN models [\[Wng24\]](../citations.rst):

- The task of explaining a prediction $y_\text{explain}$ for a multi-class model is reduced to combining the explanations for the binary prediction of $y_\text{explain}$ versus $c$ for all $c \neq y_\text{train}$:
  $$
    \nu(S) = \frac{1}{|C|-1} \sum_{c\in C \setminus \{y_\text{explain}\}} \nu_c(S),
  $$
  where $\nu_c$ is the utility function for the binary classficiation sub-task, as defined below.

- We consider a binary utility function, meaning that for a non-empty coalition $\emptyset \neq S \subseteq D$, we define its utility in prediciting $y_\text{explain}$ versus $c$ as
  $$
    \nu_c(S) = \chi \left[
        \sum_{j=1}^{k} w_{\alpha_{S,j,c}} \, \chi(y_{\alpha_{S,j,c}} = y_\text{explain})
        \geq
        \sum_{j=1}^{k} w_{\alpha_{S,j,c}} \, \chi(y_{\alpha_{S,j,c}} = c)
    \right],
  $$
  and set $\nu(\emptyset) = 0$. Note that here we define $\alpha_{S,j,c}$ to be the index of the $j$-nearest training point to $x_\text{explain}$ among all points in $D$ that are **relevant to the current binary prediction task '$y_\text{explain}$ versus $c$'**, that is, all $x_i$ for which $y_i \in \{c, y_\text{explain} \}$.

- The weights $w_i$ are normalized to the range $[0, 1]$ and then discretized to $b$ bits, meaning they are rounded to the nearest of $2^b + 1$ equally spaced values in the interval $[0, 1]$. This parameter $b$ allows the user to make a trade-off between explanation accuracy and performance.